In [ ]:
import torch

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MyMultiHeadAttention(nn.Module):
    """从零实现 Multi-Head Attention"""

    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # TODO: 定义 Q, K, V 的线性投影层
        # self.W_q = ...
        # self.W_k = ...
        # self.W_v = ...
        # self.W_o = ...

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, causal_mask=False):
        """
        Args:
            x: (batch, seq_len, d_model)
            causal_mask: 是否应用 causal mask
        Returns:
            output: (batch, seq_len, d_model)
        """
        B, N, D = x.shape

        # TODO Step 1: 线性投影得到 Q, K, V (each: [B, N, d_model])
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # TODO Step 2: 拆分为多个 head
        # reshape 为 [B, num_heads, N, head_dim]
        Q = Q.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)  # [B, num_heads, N, head_dim]
        K = K.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)  # [B, num_heads, N, head_dim]
        V = V.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)  # [B, num_heads, N, head_dim]

        # TODO Step 3: 计算 scaled dot-product attention
        # scores = Q @ K^T / sqrt(head_dim)
        # 如果 causal_mask=True, 应用下三角 mask
        # attention_weights = softmax(scores)
        # attention_output = weights @ V
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)  # [B, num_heads, N, N]
        if causal_mask:
            mask = torch.tril(torch.ones(N, N)).to(scores.device)  # [N, N]
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = F.softmax(scores, dim=-1)  # [B, num_heads, N, N]
        attention_output = torch.matmul(attention_weights, V)  # [B, num_heads, N, head_dim]

        # TODO Step 4: 合并所有 head (concat)
        # reshape 回 [B, N, d_model]
        attention_output = attention_output.transpose(1, 2).contiguous().view(B, N, D)  # [B, N, d_model]

        # TODO Step 5: 输出投影
        # output = self.W_o(concatenated)
        output = self.W_o(attention_output)

        return output

def verify_implementation():
    """验证与 PyTorch 官方 MHA 输出一致"""
    torch.manual_seed(42)
    d_model, num_heads = 64, 8
    batch_size, seq_len = 2, 10

    x = torch.randn(batch_size, seq_len, d_model)

    # 初始化自己的实现
    my_mha = MyMultiHeadAttention(d_model, num_heads)
    my_mha.eval()

    # 初始化官方实现，并同步权重
    official_mha = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
    official_mha.eval()

    # 同步权重: nn.MultiheadAttention 的 in_proj_weight = [W_q; W_k; W_v]
    with torch.no_grad():
        official_mha.in_proj_weight.copy_(
            torch.cat([my_mha.W_q.weight, my_mha.W_k.weight, my_mha.W_v.weight], dim=0)
        )
        official_mha.in_proj_bias.copy_(
            torch.cat([my_mha.W_q.bias, my_mha.W_k.bias, my_mha.W_v.bias], dim=0)
        )
        official_mha.out_proj.weight.copy_(my_mha.W_o.weight)
        official_mha.out_proj.bias.copy_(my_mha.W_o.bias)

    # 前向计算
    with torch.no_grad():
        my_output = my_mha(x, causal_mask=False)
        # 官方实现: self-attention 时 q=k=v=x
        official_output, _ = official_mha(x, x, x)

    # 对比
    max_diff = (my_output - official_output).abs().max().item()
    print(f"最大差异: {max_diff:.2e}")
    assert max_diff < 1e-5, f"差异过大: {max_diff}"
    print("✓ 验证通过! 自定义实现与 PyTorch 官方 MHA 输出一致。")

verify_implementation()

最大差异: 5.96e-08
✓ 验证通过! 自定义实现与 PyTorch 官方 MHA 输出一致。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time


class SimpleTransformerLayer(nn.Module):
    """简化的 Transformer Layer (只有 Self-Attention, 没有 FFN)
    
    支持两种模式:
      - Prefill: 一次性处理整个 prompt, 建立 KV Cache
      - Decode:  逐 token 生成, 复用 KV Cache 避免重复计算
    """

    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, kv_cache=None):
        """
        Args:
            x: [B, L, d_model] - 新 token(s) 的 embedding
            kv_cache: None 或 (k_cache, v_cache), 每个 shape [B, num_heads, past_len, head_dim]
        Returns:
            output: [B, L, d_model]
            new_kv_cache: (k, v) 更新后的 cache
        """
        B, L, _ = x.shape

        # Step 1: 线性投影 + reshape 为多头
        q = self.W_q(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)  # [B, H, L, D]
        k = self.W_k(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)  # [B, H, L, D]
        v = self.W_v(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)  # [B, H, L, D]

        # Step 2: 拼接历史 KV Cache
        if kv_cache is not None:
            k_cache, v_cache = kv_cache
            k = torch.cat([k_cache, k], dim=2)  # [B, H, past_len + L, D]
            v = torch.cat([v_cache, v], dim=2)
        new_kv_cache = (k.detach(), v.detach())

        # Step 3: Scaled Dot-Product Attention
        total_len = k.size(2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # [B, H, L, total_len]

        # Causal mask (仅 prefill 时需要; decode 时 L=1, 天然只能看到过去)
        if L > 1:
            mask = torch.triu(torch.ones(L, total_len, device=x.device), diagonal=total_len - L + 1).bool()
            scores = scores.masked_fill(mask, float('-inf'))

        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)  # [B, H, L, D]

        # Step 4: 合并多头 + 输出投影
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)  # [B, L, d_model]
        out = self.W_o(out)
        return out, new_kv_cache


# ============================================================
# 生成函数: 有 Cache vs 无 Cache
# ============================================================

def generate_with_cache(model, prompt_embeddings, num_new_tokens):
    """使用 KV Cache 的自回归生成 (Prefill + Decode)"""
    outputs = []
    with torch.no_grad():
        # Prefill: 一次处理整个 prompt, 建立 KV Cache
        out, kv_cache = model(prompt_embeddings)
        current_input = out[:, -1:, :]
        outputs.append(current_input)

        # Decode: 逐 token 生成, 每步只计算 1 个新 token
        for _ in range(num_new_tokens - 1):
            current_input, kv_cache = model(current_input, kv_cache=kv_cache)
            outputs.append(current_input)

    return torch.cat(outputs, dim=1)  # [B, num_new_tokens, d_model]


def generate_without_cache(model, prompt_embeddings, num_new_tokens):
    """无 Cache 的自回归生成 (每步重新计算完整 attention)"""
    with torch.no_grad():
        current_input = prompt_embeddings
        for _ in range(num_new_tokens):
            output, _ = model(current_input)
            current_input = torch.cat([current_input, output[:, -1:, :]], dim=1)

    return current_input[:, prompt_embeddings.size(1):, :]  # [B, num_new_tokens, d_model]


# ============================================================
# 验证正确性 + 性能对比 (支持多种 dtype)
# ============================================================

def benchmark(dtype=torch.float32, prompt_len=512, gen_len=1024):
    """运行一次完整的 benchmark"""
    torch.manual_seed(42)
    d_model, num_heads = 64, 8

    model = SimpleTransformerLayer(d_model, num_heads).cuda().to(dtype).eval()
    prompt = torch.randn(1, prompt_len, d_model, device='cuda', dtype=dtype)

    # 有 Cache
    torch.cuda.synchronize()
    t0 = time.time()
    output_with_cache = generate_with_cache(model, prompt, gen_len)
    torch.cuda.synchronize()
    cache_time = time.time() - t0

    # 无 Cache
    torch.cuda.synchronize()
    t0 = time.time()
    output_without_cache = generate_without_cache(model, prompt, gen_len)
    torch.cuda.synchronize()
    no_cache_time = time.time() - t0

    # KV Cache 显存: 2(K+V) * num_heads * total_seq_len * head_dim * bytes_per_element
    elem_size = torch.tensor([], dtype=dtype).element_size()
    kv_mem_bytes = 2 * num_heads * (prompt_len + gen_len) * (d_model // num_heads) * elem_size

    # 正确性 (低精度用更大的 tolerance)
    atol = {torch.float32: 1e-5, torch.float16: 1e-2, torch.bfloat16: 1e-1}.get(dtype, 5e-1)
    correct = torch.allclose(
        output_with_cache.float(), output_without_cache.float(), atol=atol
    )

    return {
        "dtype": dtype,
        "cache_time": cache_time,
        "no_cache_time": no_cache_time,
        "speedup": no_cache_time / cache_time,
        "kv_mem_kb": kv_mem_bytes / 1024,
        "correct": correct,
    }


# ===== 运行多 dtype 对比 =====
dtype_list = [torch.float32, torch.bfloat16, torch.float16]
prompt_len, gen_len = 512, 1024

print("=" * 72)
print(f"  KV Cache 性能对比 | prompt={prompt_len}, gen={gen_len} tokens")
print("=" * 72)
print(f"  {'dtype':<14} {'有Cache(s)':<12} {'无Cache(s)':<12} {'加速比':<8} {'KV显存':<10} {'正确性'}")
print("-" * 72)

for dtype in dtype_list:
    try:
        r = benchmark(dtype=dtype, prompt_len=prompt_len, gen_len=gen_len)
        status = "[PASS]" if r["correct"] else "[FAIL]"
        dtype_name = str(dtype).replace("torch.", "")
        print(f"  {dtype_name:<14} {r['cache_time']:<12.4f} {r['no_cache_time']:<12.4f} "
              f"{r['speedup']:<8.1f} {r['kv_mem_kb']:<8.1f}KB {status}")
    except Exception as e:
        dtype_name = str(dtype).replace("torch.", "")
        print(f"  {dtype_name:<14} SKIP ({type(e).__name__}: {str(e)[:45]})")

print("=" * 72)
print("\n  注: fp8 需要 CUDA compute capability >= 8.9 (Ada/Hopper) 且 PyTorch >= 2.1")

In [ ]:
import torch
import torch.nn as nn
import time


# ============================================================
# CUDA Graph 教学示例
# ============================================================
# 核心思想: 将多个小 kernel 的 launch 开销合并为一次 graph replay
#
# 正常执行: CPU 逐个发射 kernel → GPU 执行 (每次都有 launch latency)
# CUDA Graph: 一次录制所有 kernel → 之后 replay 只需一次 launch
#
# 适用场景: decode 阶段 (计算量小, kernel launch 开销占比高)
# 不适用: 动态 shape (如 prefill 阶段 seq_len 变化)
# ============================================================


class SmallMLP(nn.Module):
    """一个小模型, 用来演示 CUDA Graph 对 kernel launch 密集型场景的加速"""
    def __init__(self, d_model=256, hidden=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, d_model),
        )

    def forward(self, x):
        return self.net(x)


# ===== 配置 =====
batch_size, seq_len, d_model = 1, 1, 256  # 模拟 decode: 每次只处理 1 个 token
num_iterations = 1000

model = SmallMLP(d_model).cuda().eval()

# ============================================================
# 方法 1: 普通执行 (每次都经历 kernel launch 开销)
# ============================================================

x_normal = torch.randn(batch_size, seq_len, d_model, device='cuda')

# Warmup
for _ in range(10):
    _ = model(x_normal)
torch.cuda.synchronize()

t0 = time.time()
for _ in range(num_iterations):
    out_normal = model(x_normal)
torch.cuda.synchronize()
normal_time = time.time() - t0


# ============================================================
# 方法 2: CUDA Graph (录制一次, 反复 replay)
# ============================================================

# Step 0: 分配 static tensor (地址在 graph 生命周期内不能变!)
static_input = torch.randn(batch_size, seq_len, d_model, device='cuda')

# Warmup (确保所有 lazy 初始化完成: cuBLAS handle, memory pool 等)
with torch.no_grad():
    for _ in range(3):
        _ = model(static_input)
torch.cuda.synchronize()

# Step 1: Capture - 录制计算图
graph = torch.cuda.CUDAGraph()
with torch.no_grad():
    with torch.cuda.graph(graph):
        # 这里的操作不会真正执行, 只是被录制
        static_output = model(static_input)

# Step 2: Replay - 用新数据重放
torch.cuda.synchronize()
t0 = time.time()
for i in range(num_iterations):
    # 必须用 copy_ 原地更新输入 (不能重新分配 tensor!)
    static_input.copy_(torch.randn(batch_size, seq_len, d_model, device='cuda'))
    # 一次 replay = 执行录制时的所有 kernel
    graph.replay()
torch.cuda.synchronize()
graph_time = time.time() - t0

# Step 3: 读取结果 (结果始终写入 static_output 的固定地址)
final_output = static_output.clone()


# ============================================================
# 结果对比
# ============================================================

print("=" * 60)
print(f"  CUDA Graph 性能对比 | {num_iterations} iterations")
print(f"  Model: MLP ({d_model}→1024→1024→{d_model})")
print(f"  Input: [{batch_size}, {seq_len}, {d_model}] (模拟 decode)")
print("=" * 60)
print(f"  普通执行:     {normal_time*1000:.2f} ms  ({normal_time/num_iterations*1e6:.1f} μs/iter)")
print(f"  CUDA Graph:   {graph_time*1000:.2f} ms  ({graph_time/num_iterations*1e6:.1f} μs/iter)")
print(f"  加速比:       {normal_time/graph_time:.2f}x")
print("-" * 60)
print(f"  输出 shape:   {list(final_output.shape)}")
print("=" * 60)

# 验证正确性: graph replay 的结果和普通执行一致
with torch.no_grad():
    test_input = torch.randn(batch_size, seq_len, d_model, device='cuda')
    expected = model(test_input)
    static_input.copy_(test_input)
    graph.replay()
    torch.cuda.synchronize()
    assert torch.allclose(static_output, expected, atol=1e-5), "CUDA Graph 输出不一致!"
    print("\n  [PASS] CUDA Graph 输出与普通执行一致")

print("""
  ┌─────────────────────────────────────────────────────────┐
  │ CUDA Graph 使用注意事项:                                │
  │                                                         │
  │ 1. 输入必须用 .copy_() 原地更新, 不能重新分配 tensor   │
  │ 2. 录制期间不能有动态 shape / 条件分支 / CPU 同步      │
  │ 3. 不能包含 CPU-GPU 同步操作 (如 .item(), print)       │
  │ 4. 适合 decode 阶段 (固定 shape, 小计算, 多 kernel)    │
  │ 5. 不适合 prefill (seq_len 变化) 或含动态控制流的模型  │
  └─────────────────────────────────────────────────────────┘
""")

In [5]:
import torch
import torch.nn as nn
import time

# ============================================================
# Profiling 教学: 判断 kernel launch 是否是瓶颈
# ============================================================
# 方法: 用 torch.profiler 对比 CPU time vs CUDA time
#   - CPU time 包含 kernel launch 开销 + Python overhead
#   - CUDA time (device time) 是 GPU 上实际执行时间
#   - 如果 CPU time >> CUDA time → launch bound (CUDA Graph 有用)
#   - 如果 CPU time ≈ CUDA time → compute bound (CUDA Graph 没用)
# ============================================================


class DeepNarrowMLP(nn.Module):
    """很多小 layer → 很多小 kernel → launch bound"""
    def __init__(self, d=64, num_layers=20):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(d, d) for _ in range(num_layers)])

    def forward(self, x):
        for layer in self.layers:
            x = torch.relu(layer(x))
        return x


class ShallowWideMLP(nn.Module):
    """少量大 layer → 少量大 kernel → compute bound"""
    def __init__(self, d=2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, d),
            nn.ReLU(),
            nn.Linear(d, d),
        )

    def forward(self, x):
        return self.net(x)


def profile_model(model, x, name, num_steps=50):
    """Profile 一个模型, 打印 CPU vs Device(CUDA) 时间"""
    model.eval()

    # Warmup
    for _ in range(10):
        _ = model(x)
    torch.cuda.synchronize()

    # Profile
    with torch.profiler.profile(
        activities=[
            torch.profiler.ProfilerActivity.CPU,
            torch.profiler.ProfilerActivity.CUDA,
        ],
        record_shapes=True,
    ) as prof:
        for _ in range(num_steps):
            _ = model(x)
        torch.cuda.synchronize()

    # 汇总 (PyTorch 2.12+ 用 self_device_time_total, 旧版用 cuda_time_total)
    events = prof.key_averages()
    total_cpu = sum(e.self_cpu_time_total for e in events) / 1000  # ms
    total_device = sum(e.self_device_time_total for e in events) / 1000  # ms

    ratio = total_cpu / total_device if total_device > 0 else float('inf')
    bottleneck = "LAUNCH BOUND (CUDA Graph 有用!)" if ratio > 2.0 else "COMPUTE BOUND (CUDA Graph 收益小)"

    print(f"\n  [{name}]")
    print(f"  CPU total:    {total_cpu:.2f} ms")
    print(f"  Device total: {total_device:.2f} ms")
    print(f"  CPU/Device:   {ratio:.2f}x")
    print(f"  判断:         {bottleneck}")

    return events


# ===== 场景 1: 很多小 kernel (decode-like) =====
print("=" * 60)
print("  Profiling: 判断 kernel launch 是否是瓶颈")
print("=" * 60)

model_deep = DeepNarrowMLP(d=64, num_layers=20).cuda()
x_small = torch.randn(1, 1, 64, device='cuda')
events_deep = profile_model(model_deep, x_small, "20 层小 Linear (d=64, input=[1,1,64])")

# ===== 场景 2: 少量大 kernel (training-like) =====
model_wide = ShallowWideMLP(d=2048).cuda()
x_big = torch.randn(32, 128, 2048, device='cuda')
events_wide = profile_model(model_wide, x_big, "2 层大 Linear (d=2048, input=[32,128,2048])")

print("\n" + "=" * 60)
print("  Top-5 kernels (场景 1: 小 kernel)")
print("-" * 60)
print(events_deep.table(sort_by="self_device_time_total", row_limit=5))

print("=" * 60)
print("  Top-5 kernels (场景 2: 大 kernel)")
print("-" * 60)
print(events_wide.table(sort_by="self_device_time_total", row_limit=5))

  Profiling: 判断 kernel launch 是否是瓶颈

  [20 层小 Linear (d=64, input=[1,1,64])]
  CPU total:    31.12 ms
  Device total: 6.93 ms
  CPU/Device:   4.49x
  判断:         LAUNCH BOUND (CUDA Graph 有用!)

  [2 层大 Linear (d=2048, input=[32,128,2048])]
  CPU total:    161.84 ms
  Device total: 318.70 ms
  CPU/Device:   0.51x
  判断:         COMPUTE BOUND (CUDA Graph 收益小)

  Top-5 kernels (场景 1: 小 kernel)
------------------------------------------------------------
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ---------

In [7]:
import torch
import time
import triton
import triton.language as tl

@triton.jit
def vector_add_kernel(
    x_ptr,
    y_ptr,
    output_ptr,
    N,
    BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < N
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    out = tl.add(x, y)
    tl.store(output_ptr + offsets, out, mask=mask)


def vector_add(x: torch.Tensor, y: torch.Tensor, block_size=1024) -> torch.Tensor:
    assert x.shape == y.shape
    assert x.is_cuda and y.is_cuda
    N = x.numel()
    output = torch.empty_like(x)
    grid = ((N + block_size - 1) // block_size,)
    vector_add_kernel[grid](x, y, output, N, BLOCK_SIZE=block_size)
    return output


# ===== 验证 + Benchmark =====
N = 10_000_000  # 用大向量, 让 GPU 计算时间主导而非 launch overhead
x = torch.randn(N, device='cuda')
y = torch.randn(N, device='cuda')
iteration = 1000

# Warmup
for _ in range(10):
    _ = x + y
    _ = vector_add(x, y)

# Native benchmark (正确使用 cuda.synchronize)
torch.cuda.synchronize()
start_time = time.time()
for _ in range(iteration):
    output_native = x + y
torch.cuda.synchronize()
native_elapsed = time.time() - start_time
print(f"Native: {native_elapsed * 1000:.3f} ms")

# Triton benchmark
for block_size in [256, 512, 1024, 2048]:
    torch.cuda.synchronize()
    start_time = time.time()
    for _ in range(iteration):
        vector_add_result = vector_add(x, y, block_size)
    torch.cuda.synchronize()
    triton_elapsed = time.time() - start_time
    print(f"Triton (BLOCK_SIZE={block_size:>4}): {triton_elapsed * 1000:.3f} ms  ({native_elapsed/triton_elapsed:.2f}x vs native)")

# 正确性验证
print(f"\nMax error: {(x + y - vector_add_result).abs().max().item():.2e}")

# 也可以用 triton.testing.do_bench 获得更精确的单次 kernel 时间
ms_native = triton.testing.do_bench(lambda: x + y)
ms_triton = triton.testing.do_bench(lambda: vector_add(x, y))
print(f"\n[do_bench] Native: {ms_native:.3f} ms, Triton: {ms_triton:.3f} ms, ratio: {ms_native/ms_triton:.2f}x")

Native: 73.578 ms
Triton (BLOCK_SIZE= 256): 72.137 ms  (1.02x vs native)
Triton (BLOCK_SIZE= 512): 70.559 ms  (1.04x vs native)
Triton (BLOCK_SIZE=1024): 67.942 ms  (1.08x vs native)
Triton (BLOCK_SIZE=2048): 67.779 ms  (1.09x vs native)

Max error: 0.00e+00

[do_bench] Native: 0.090 ms, Triton: 0.090 ms, ratio: 1.00x


In [ ]:

import torch
import time
import numpy as np

# def benchmark_gemm_cuda_events(M, N, K, num_warmup=10, num_runs=100):
#     """
#     使用 CUDA Events 精确测量 GEMM 执行时间

#     Args:
#         M, N, K: 矩阵维度 C[M,N] = A[M,K] x B[K,N]
#         num_warmup: warmup 迭代次数
#         num_runs: 测量迭代次数
#     Returns:
#         avg_ms: 平均执行时间 (毫秒)
#         std_ms: 标准差 (毫秒)
#         tflops: 达到的 TFLOPS
#     """
#     A = torch.randn(M, K, device='cuda', dtype=torch.float8_e5m2)
#     B = torch.randn(K, N, device='cuda', dtype=torch.float8_e5m2)

#     for _ in range(num_warmup):
#         _ = torch.mm(A, B)  # Warmup iterations

#     start_event = torch.cuda.Event(enable_timing=True)
#     end_event = torch.cuda.Event(enable_timing=True)

#     times = []
#     for _ in range(num_runs):
#         start_event.record()
#         C = torch.mm(A, B)  # GEMM
#         end_event.record()
#         torch.cuda.synchronize()
#         times.append(start_event.elapsed_time(end_event))

#     avg_ms = sum(times) / len(times)
#     std_ms = np.std(times)
#     flops = 2 * M * N * K
#     tflops = flops / (avg_ms * 1e-3) / 1e12

#     return avg_ms, std_ms, tflops


def benchmark_gemm_cuda_events(M, N, K, num_warmup=10, num_runs=100):
    A = torch.randn(M, K, device='cuda', dtype=torch.bfloat16).to(torch.float8_e4m3fn)
    B = torch.randn(N, K, device='cuda', dtype=torch.bfloat16).to(torch.float8_e4m3fn).T

    scale_a = torch.tensor(1.0, device='cuda', dtype=torch.float32)
    scale_b = torch.tensor(1.0, device='cuda', dtype=torch.float32)

    # Warmup
    for _ in range(num_warmup):
        _ = torch._scaled_mm(A, B, scale_a=scale_a, scale_b=scale_b, out_dtype=torch.bfloat16)

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    times = []
    for _ in range(num_runs):
        start_event.record()
        C = torch._scaled_mm(A, B, scale_a=scale_a, scale_b=scale_b, out_dtype=torch.bfloat16)
        end_event.record()
        torch.cuda.synchronize()
        times.append(start_event.elapsed_time(end_event))

    avg_ms = sum(times) / len(times)
    std_ms = np.std(times)
    flops = 2 * M * N * K
    tflops = flops / (avg_ms * 1e-3) / 1e12

    return avg_ms, std_ms, tflops


# ===== 测试不同矩阵大小 =====
sizes = [32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384]

for size in sizes:
    avg_ms, std_ms, tflops = benchmark_gemm_cuda_events(size, size, size, 10, 100)
    print(f"Size {size}: avg_time = {avg_ms:.2f} ms, std_time = {std_ms:.2f} ms, TFLOPS = {tflops:.2f}")

RuntimeError: Invalid scaling configuration.
- For TensorWise scaling, a and b should be float8, scales should be float and singletons.
- For RowWise scaling, a and b should be float8, scales should be float, scale_a should be (32, 1) and scale_b should be (1, 32), and both should be contiguous.
- For BlockWise 1x128 scaling, a and b should be float8, scales should be float, scale_a should be (32, 1) and scale_b should be (1, 32), and both should be outer-dim-major.
- For BlockWise 128x128 scaling, a and b should be float8, scales should be float, scale_a should be (1, 1) and scale_b should be (1, 1), and both should be near-inner-dim-major (with 16-byte aligned strides).
- For Blockwise 1x32 scaling, a and b should be float8, scales should be float8_e8m0fnu, scale_a should have 512 elements and scale_b should have 512 elements, and both should be contiguous.
- For Blockwise 1x16 scaling, a and b should be float4 (packed 2x), scales should be float8_e4m3fn, scale_a should have 512 elem